In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score

# hc/k_B in cm*K when vibration frequencies are given in cm^-1
hc_k = 1.4388

# Temperature range: 273-473 K with step 10 K
T = np.arange(273, 474, 10, dtype=float)

# NIST vibration frequencies (cm^-1)
# H2O has 3 normal modes, HDO has 3 normal modes,
# H2 has 1 mode, HD has 1 mode.
nu_h2o = np.array([1595.0, 3657.0, 3756.0], dtype=float)
nu_hdo = np.array([1402.0, 2727.0, 3707.0], dtype=float)
nu_h2 = np.array([4161.0], dtype=float)
nu_hd = np.array([3632.0], dtype=float)

In [ ]:
def calc_u(nu, T):
    """Calculate dimensionless reduced frequency u = (hc/k_B) * nu / T."""
    nu = np.asarray(nu, dtype=float)
    T = np.asarray(T, dtype=float)
    return hc_k * nu / T[..., np.newaxis]


def calc_rpfr(nu_light, nu_heavy, T):
    """
    Calculate RPFR (f) using Urey-Bigeleisen-Mayer expression:
    f = prod_i [ (nu'_i/nu_i) * exp(-u'_i/2)/(1-exp(-u'_i)) * (1-exp(-u_i))/exp(-u_i/2) ]
    """
    nu_light = np.asarray(nu_light, dtype=float)
    nu_heavy = np.asarray(nu_heavy, dtype=float)

    if nu_light.size == 0 or nu_heavy.size == 0:
        raise ValueError("Frequency arrays cannot be empty. Please fill NIST frequencies first.")
    if nu_light.shape != nu_heavy.shape:
        raise ValueError("nu_light and nu_heavy must have the same number of vibrational modes.")

    u_light = calc_u(nu_light, T)
    u_heavy = calc_u(nu_heavy, T)

    ratio_nu = nu_heavy / nu_light
    term_heavy = np.exp(-u_heavy / 2.0) / (1.0 - np.exp(-u_heavy))
    term_light = (1.0 - np.exp(-u_light)) / np.exp(-u_light / 2.0)

    # Broadcast ratio_nu over temperature axis, then take product over vibrational modes.
    rpfr_terms = ratio_nu * term_heavy * term_light
    f = np.prod(rpfr_terms, axis=-1)
    return f

In [ ]:
f_water = calc_rpfr(nu_h2o, nu_hdo, T)
f_hydrogen = calc_rpfr(nu_h2, nu_hd, T)

alpha = f_water / f_hydrogen
ln_alpha = np.log(alpha)

print("| 温度 (K) | $H_2O$ 配分函数比 ($f_{H_2O}$) | $H_2$ 配分函数比 ($f_{H_2}$) | 分馏系数 ($\\ln\\alpha$) |")
print("| :---: | :---: | :---: | :---: |")
for Ti, fw, fh, lai in zip(T, f_water, f_hydrogen, ln_alpha):
    print(f"| {Ti:.1f} | {fw:.5f} | {fh:.5f} | {lai:.5f} |")

In [ ]:
def model_1(T, A, B):
    return A + B / T


def model_2(T, A, B):
    return A + B / (T ** 2)


params1, _ = curve_fit(model_1, T, ln_alpha)
params2, _ = curve_fit(model_2, T, ln_alpha)
A1, B1 = params1
A2, B2 = params2

ln_alpha_pred1 = model_1(T, A1, B1)
ln_alpha_pred2 = model_2(T, A2, B2)

r2_1 = r2_score(ln_alpha, ln_alpha_pred1)
r2_2 = r2_score(ln_alpha, ln_alpha_pred2)
rmse_1 = np.sqrt(np.mean((ln_alpha - ln_alpha_pred1) ** 2))
rmse_2 = np.sqrt(np.mean((ln_alpha - ln_alpha_pred2) ** 2))

print("Model 1: ln(alpha) = A + B/T")
print(f"A1 = {A1:.8e}, B1 = {B1:.8e}")
print(f"R^2 = {r2_1:.8f}, RMSE = {rmse_1:.8e}")

print("\nModel 2: ln(alpha) = A + B/T^2")
print(f"A2 = {A2:.8e}, B2 = {B2:.8e}")
print(f"R^2 = {r2_2:.8f}, RMSE = {rmse_2:.8e}")

if (r2_1 > r2_2 and rmse_1 <= rmse_2) or (r2_1 >= r2_2 and rmse_1 < rmse_2):
    better_model = "Model 1: ln(alpha) = A + B/T"
elif (r2_2 > r2_1 and rmse_2 <= rmse_1) or (r2_2 >= r2_1 and rmse_2 < rmse_1):
    better_model = "Model 2: ln(alpha) = A + B/T^2"
else:
    # If metrics disagree, prioritize R^2 and report tie-break rationale.
    better_model = "Model 1: ln(alpha) = A + B/T" if r2_1 >= r2_2 else "Model 2: ln(alpha) = A + B/T^2"

print(f"\nMore accurate formula based on fit metrics: {better_model}")

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(T, ln_alpha, label="UBM Calculation", s=40)
plt.plot(T, ln_alpha_pred1, "-", linewidth=2, label=f"Model 1: A + B/T (R^2={r2_1:.5f})")
plt.plot(T, ln_alpha_pred2, "--", linewidth=2, label=f"Model 2: A + B/T^2 (R^2={r2_2:.5f})")

plt.xlabel("Temperature (K)")
plt.ylabel(r"$\ln\alpha$")
plt.legend()
plt.tight_layout()
plt.show()